# DDoS-SDN — complete experiment notebook (revision R2)
One run produces every DDoS-SDN table and figure in the manuscript.

| Section | Content | Paper item |
|---|---|---|
| 0 | Environment (hardware, software) | Section 3.2 |
| 1 | Dataset audit and input-feature list | Section 3.1, new audit table |
| 2 | Descriptive statistics, correlation | Table 6, Figure 7 |
| 3 | Mutual-information ranking | Figures 3–6 |
| 4–6 | Six classifiers on Raw, MI Top-5, MI Top-12 (reports, confusion matrices, ROC) | Tables 7–9, Figures 8–9 |
| 7 | Raw vs Top-5 vs Top-12 | Tables 8–9, Figures 10–17 |
| 8 | 10-fold CV (SVM excluded for runtime) | Tables 10–12, Figure 25 |
| 9 | McNemar, all pairs, Holm | Table 15 |
| 10 | Timing (training / inference, mean ± SD) | Tables 16/18, Figures 23–24 |
| 11 | Unconstrained DT / RF / XGBoost | Table 20 |
| 12 | Robustness: time-ordered, unseen hosts, no topology | new robustness table/figure |
| 13 | Pipeline figure, export (Excel, paper map, zip) | Figure 2 |

In [1]:
# ===================== CONFIGURATION =====================
import os
DATASET = "DDoS-SDN"
DATA_PATH = os.environ.get("SDN_PATH", "/kaggle/input/datasets/aikenkazin/ddos-sdn-dataset/dataset_sdn.csv")
OUT = os.environ.get("SDN_OUT", "/kaggle/working/DDoS_SDN_results")
TARGET = "label"
SEED = 42
N_JOBS = os.cpu_count() or 1          # same thread count for every model; reported in Section 0
TRAIN_REPEATS = 3                     # training repeats per model (SVM: 1, to save time)
INFER_REPEATS = 10                    # inference repeats per model
RUN_CV = True
CV_FOLDS = 10
ID_COLS = ["dt", "src", "dst"]        # never model inputs; used only to define robustness splits
TOPOLOGY_COLS = ["switch", "port_no"]
FEATURE_SETS = {"RAW": None, "MI Top-5": 5, "MI Top-12": 12}
CLASS_NAMES = ["Benign", "Attack"]
N_CLASSES = 2

In [2]:
# ===================== SHARED CODE =====================
# ---------------------------------------------------------------------
# SHARED CODE (identical in the DDoS-SDN and APA-DDoS notebooks)
# ---------------------------------------------------------------------
import os, sys, time, json, platform, itertools, warnings, zipfile
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import chi2, binomtest

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             roc_curve, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "font.size": 10})

DIRS = ["00_environment", "01_dataset_audit", "02_descriptive", "03_mutual_information",
        "04_raw", "05_mi_top5", "06_mi_top12", "07_comparison", "08_cross_validation",
        "09_mcnemar", "10_timing", "11_capacity_check", "12_dataset_specific", "13_export"]
FS_DIR = {"RAW": "04_raw", "MI Top-5": "05_mi_top5", "MI Top-12": "06_mi_top12"}
TABLES = {}          # name -> DataFrame, exported to one Excel file at the end
PAPER_MAP = []       # (output file, paper item, description)


def setup_dirs():
    for d in DIRS:
        os.makedirs(os.path.join(OUT, d), exist_ok=True)


def p(d, name):
    return os.path.join(OUT, d, name)


def save_table(df, d, name, paper="", desc="", index=False):
    path = p(d, name + ".csv")
    df.to_csv(path, index=index)
    TABLES[name[:31]] = df.reset_index() if index else df
    PAPER_MAP.append((os.path.relpath(path, OUT), paper, desc))
    return path


def save_fig(fig, d, name, paper="", desc=""):
    path = p(d, name + ".png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    PAPER_MAP.append((os.path.relpath(path, OUT), paper, desc))
    return path


# ---------------- environment ----------------
def environment():
    cpu = platform.processor() or ""
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    cpu = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    ram = ""
    try:
        with open("/proc/meminfo") as f:
            ram = f"{int(f.readline().split()[1]) / 1024**2:.1f} GB"
    except Exception:
        pass
    env = pd.DataFrame([
        ("CPU", cpu), ("Logical CPUs", os.cpu_count()), ("RAM", ram), ("OS", platform.platform()),
        ("Python", platform.python_version()), ("scikit-learn", sklearn.__version__),
        ("XGBoost", xgboost.__version__), ("NumPy", np.__version__), ("pandas", pd.__version__),
        ("Threads per model (n_jobs)", N_JOBS), ("Device", "CPU only; no GPU used by any classifier"),
        ("Random seed", SEED), ("Training-time repeats", TRAIN_REPEATS),
        ("Inference-time repeats", INFER_REPEATS),
    ], columns=["Item", "Value"])
    save_table(env, "00_environment", "environment", "Section 3.2 / Table of setup",
               "Hardware and software used for every timing result")
    print(env.to_string(index=False))
    return env


# ---------------- models ----------------
def models_main():
    """Section 3.2 hyper-parameters (fixed a priori)."""
    lr_solver = "liblinear" if N_CLASSES == 2 else "lbfgs"
    return {
        "Logistic Regression": LogisticRegression(solver=lr_solver, C=0.03, max_iter=1000,
                                                  random_state=SEED),
        "SVM": SVC(kernel="rbf", C=1.0, probability=False, random_state=SEED),
        "KNN": KNeighborsClassifier(n_neighbors=5, metric="manhattan", weights="distance",
                                    n_jobs=N_JOBS),
        "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=8, max_leaf_nodes=11,
                                                random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=20, criterion="entropy", max_depth=10,
                                                min_samples_split=100, min_samples_leaf=20,
                                                max_features="sqrt", random_state=SEED,
                                                n_jobs=N_JOBS),
        "XGBoost": XGBClassifier(eval_metric="logloss" if N_CLASSES == 2 else "mlogloss",
                                 tree_method="hist", random_state=SEED, n_jobs=N_JOBS),
    }


def models_unconstrained():
    return {
        "Decision Tree (unconstrained)": DecisionTreeClassifier(random_state=SEED),
        "Random Forest (unconstrained)": RandomForestClassifier(n_estimators=200, criterion="entropy",
                                                                max_features="sqrt", random_state=SEED,
                                                                n_jobs=N_JOBS),
        "XGBoost (500 trees, depth 10)": XGBClassifier(n_estimators=500, max_depth=10,
                                                       eval_metric="logloss" if N_CLASSES == 2 else "mlogloss",
                                                       tree_method="hist", random_state=SEED, n_jobs=N_JOBS),
    }


# ---------------- pipeline ----------------
class MISelector(BaseEstimator, TransformerMixin):
    """Mutual information (sklearn mutual_info_classif: k-NN estimator, n_neighbors=3,
    all inputs treated as continuous after encoding, fixed random_state).
    Fitted on training data only; ties broken by column order (stable sort)."""
    def __init__(self, k=5, random_state=42):
        self.k, self.random_state = k, random_state
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.scores_ = mutual_info_classif(X, y, random_state=self.random_state)
        order = np.argsort(-self.scores_, kind="stable")
        self.indices_ = np.sort(order[: min(self.k, X.shape[1])])
        return self
    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.indices_]


def make_pipeline(num_cols, cat_cols, model=None, k=None):
    """impute -> ordinal-encode (unseen categories -> -1) -> [MI top-k] -> scale -> model.
    Every step is fitted on the training data (or training fold) only."""
    tr = []
    if num_cols:
        tr.append(("num", SimpleImputer(strategy="median"), list(num_cols)))
    if cat_cols:
        tr.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                           unknown_value=-1))]), list(cat_cols)))
    steps = [("prep", ColumnTransformer(tr, verbose_feature_names_out=False))]
    if k is not None:
        steps.append(("mi", MISelector(k=k, random_state=SEED)))
    steps.append(("scale", StandardScaler()))
    if model is not None:
        steps.append(("clf", model))
    return Pipeline(steps)


def selected_names(prep_pipe, num_cols, cat_cols):
    names = list(num_cols) + list(cat_cols)
    if "mi" in prep_pipe.named_steps:
        return [names[i] for i in prep_pipe.named_steps["mi"].indices_]
    return names


def model_scores(clf, Xt):
    """Class scores used for ROC/PR: probabilities if available, otherwise decision values."""
    if hasattr(clf, "predict_proba"):
        s = clf.predict_proba(Xt)
    else:
        s = clf.decision_function(Xt)
    if N_CLASSES == 2:
        return s[:, 1] if s.ndim == 2 else s
    return s


# ---------------- metrics ----------------
def ovr_auc(y, s):
    return float(np.mean([roc_auc_score((y == k).astype(int), s[:, k]) for k in range(N_CLASSES)]))


def ovr_ap(y, s):
    return float(np.mean([average_precision_score((y == k).astype(int), s[:, k]) for k in range(N_CLASSES)]))


def compute_metrics(y, pred, s):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    if N_CLASSES == 2:
        tn, fp, fn, tp = cm.ravel()
        return {"Accuracy (%)": accuracy_score(y, pred) * 100,
                "Precision (%)": precision_score(y, pred, zero_division=0) * 100,
                "Recall (%)": recall_score(y, pred, zero_division=0) * 100,
                "Specificity (%)": tn / (tn + fp) * 100 if tn + fp else np.nan,
                "F1 (%)": f1_score(y, pred, zero_division=0) * 100,
                "Weighted F1 (%)": f1_score(y, pred, average="weighted") * 100,
                "ROC-AUC": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s),
                "TN": tn, "FP": fp, "FN": fn, "TP": tp}
    spec = []
    for k in range(N_CLASSES):
        tn = cm.sum() - cm[k, :].sum() - cm[:, k].sum() + cm[k, k]; fp = cm[:, k].sum() - cm[k, k]
        spec.append(tn / (tn + fp) if tn + fp else np.nan)
    out = {"Accuracy (%)": accuracy_score(y, pred) * 100,
           "Macro precision (%)": precision_score(y, pred, average="macro", zero_division=0) * 100,
           "Macro recall (%)": recall_score(y, pred, average="macro", zero_division=0) * 100,
           "Macro specificity (%)": np.nanmean(spec) * 100,
           "Macro F1 (%)": f1_score(y, pred, average="macro", zero_division=0) * 100,
           "Micro F1 (%)": f1_score(y, pred, average="micro", zero_division=0) * 100,
           "Weighted F1 (%)": f1_score(y, pred, average="weighted", zero_division=0) * 100,
           "ROC-AUC (macro OvR)": ovr_auc(y, s), "PR-AUC (macro OvR)": ovr_ap(y, s)}
    for k, name in enumerate(CLASS_NAMES):
        out[f"Recall {name} (%)"] = cm[k, k] / cm[k, :].sum() * 100 if cm[k, :].sum() else np.nan
    return out


def acc_key():
    return "Accuracy (%)"


def f1_key():
    return "F1 (%)" if N_CLASSES == 2 else "Macro F1 (%)"


def auc_key():
    return "ROC-AUC" if N_CLASSES == 2 else "ROC-AUC (macro OvR)"


def mcnemar(y, p1, p2):
    a, b_ = (p1 == y), (p2 == y)
    b = int(np.sum(a & ~b_)); c = int(np.sum(~a & b_))
    if b + c == 0:
        return b, c, np.nan, 1.0, "no discordant pairs"
    if b + c < 25:
        return b, c, np.nan, float(binomtest(b, b + c, 0.5).pvalue), "exact binomial"
    st = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, st, float(chi2.sf(st, 1)), "chi-square with continuity correction"


def holm(pvals):
    pv = np.asarray(pvals, float); m = len(pv); order = np.argsort(pv)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * pv[i]); adj[i] = min(run, 1.0)
    return adj


# ---------------- figures ----------------
def fig_cm_single(cm, title):
    fig, ax = plt.subplots(figsize=(4.4, 3.9))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=max(cm.max(), 1))
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=10, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() / 2 else "#08306b")
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title, fontsize=10, fontweight="bold")
    fig.tight_layout()
    return fig


def fig_cm_grid(cms, suptitle):
    n = len(cms); cols = 3; rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 4.1 * rows))
    axes = np.atleast_1d(axes).ravel()
    letters = "abcdefghij"
    for i, (ax, (name, cm)) in enumerate(zip(axes, cms.items())):
        acc = np.trace(cm) / cm.sum() * 100
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=max(cm.max(), 1))
        for a in range(cm.shape[0]):
            for b in range(cm.shape[1]):
                ax.text(b, a, f"{cm[a, b]:,}", ha="center", va="center", fontsize=10, fontweight="bold",
                        color="white" if cm[a, b] > cm.max() / 2 else "#08306b")
        ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        ax.set_title(f"({letters[i]}) {name}\nAcc = {acc:.2f}%", fontsize=10, fontweight="bold")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=12); fig.tight_layout()
    return fig


def fig_roc(curves, title):
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    for name, (fpr, tpr, auc) in curves.items():
        ax.plot(fpr, tpr, lw=1.8, label=f"{name} (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(title, fontsize=11); ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def roc_points(y, s):
    if N_CLASSES == 2:
        fpr, tpr, _ = roc_curve(y, s); return fpr, tpr
    Y = np.eye(N_CLASSES)[y]                      # micro-average OvR curve for display
    fpr, tpr, _ = roc_curve(Y.ravel(), np.asarray(s).ravel()); return fpr, tpr


def fig_grouped_bars(df, value, ylabel, title, order, log=False, annotate=True, legend_title="Feature set"):
    piv = df.pivot_table(index="Model", columns="Feature set", values=value, sort=False)
    piv = piv[[c for c in order if c in piv.columns]]
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    piv.plot(kind="bar", ax=ax, logy=log, edgecolor="black", width=0.8)
    if annotate:
        for cont in ax.containers:
            ax.bar_label(cont, fmt="%.2f" if not log else "%.3g", fontsize=7, rotation=90, padding=2)
    ax.set_ylabel(ylabel); ax.set_xlabel(""); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
    ax.legend(title=legend_title, fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right"); fig.tight_layout()
    return fig


# ---------------- evaluation on a fixed split ----------------
def evaluate(X, y, tr, te, num_cols, cat_cols, model_dict, feature_sets, tag, out_dirs,
             timing=True, save_reports=True, figures=True):
    """Fit every model on the training indices, evaluate once on the test indices.
    Saves classification reports, confusion matrices and ROC curves per feature set."""
    rows, preds, selected = [], {}, {}
    yte = np.asarray(y)[te]; ytr = np.asarray(y)[tr]
    for fs, k in feature_sets.items():
        d = out_dirs[fs]
        prep = make_pipeline(num_cols, cat_cols, None, k)
        t0 = time.perf_counter(); prep.fit(X.iloc[tr], ytr); t_fit_prep = time.perf_counter() - t0
        t0 = time.perf_counter(); Xtr_t = prep.transform(X.iloc[tr]); t_tr_tf = time.perf_counter() - t0
        t0 = time.perf_counter(); Xte_t = prep.transform(X.iloc[te]); t_te_tf = time.perf_counter() - t0
        selected[fs] = selected_names(prep, num_cols, cat_cols)
        fslug = fs.replace(" ", "").replace("-", "")
        cms, curves = {}, {}
        for name, base in model_dict.items():
            reps = 1 if (not timing or name == "SVM") else TRAIN_REPEATS
            fit_t = []
            for _ in range(reps):
                clf = clone(base)
                t0 = time.perf_counter(); clf.fit(Xtr_t, ytr); fit_t.append(time.perf_counter() - t0)
            inf_t = []
            for _ in range(INFER_REPEATS if timing else 1):
                t0 = time.perf_counter(); pred = clf.predict(Xte_t); inf_t.append(time.perf_counter() - t0)
            s = model_scores(clf, Xte_t)
            m = compute_metrics(yte, pred, s)
            n = len(te)
            row = {"Split": tag, "Feature set": fs, "Model": name, "N features": len(selected[fs]), **m,
                   "Preprocessing+MI fit (s)": t_fit_prep, "Transform train+test (s)": t_tr_tf + t_te_tf,
                   "Training time mean (s)": np.mean(fit_t),
                   "Training time SD (s)": np.std(fit_t, ddof=1) if len(fit_t) > 1 else np.nan,
                   "Training repeats": len(fit_t),
                   "Inference time mean (s)": np.mean(inf_t),
                   "Inference time SD (s)": np.std(inf_t, ddof=1) if len(inf_t) > 1 else np.nan,
                   "Latency (ms/sample)": np.mean(inf_t) / n * 1000,
                   "Throughput (samples/s)": n / np.mean(inf_t), "Test samples": n}
            rows.append(row); preds[(fs, name)] = pred
            cm = confusion_matrix(yte, pred, labels=list(range(N_CLASSES))); cms[name] = cm
            fpr, tpr = roc_points(yte, s); curves[name] = (fpr, tpr, m[auc_key()])
            safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
            if save_reports:
                rep = classification_report(yte, pred, labels=list(range(N_CLASSES)), target_names=CLASS_NAMES,
                                            digits=4, zero_division=0)
                with open(p(d, f"classification_report_{tag}_{fslug}_{safe}.txt"), "w") as fh:
                    fh.write(f"{DATASET} | {tag} | {fs} | {name}\n\n{rep}")
                pd.DataFrame(classification_report(yte, pred, labels=list(range(N_CLASSES)),
                                                   target_names=CLASS_NAMES, output_dict=True,
                                                   zero_division=0)).T.round(4).to_csv(
                    p(d, f"classification_report_{tag}_{fslug}_{safe}.csv"))
                if figures:
                    save_fig(fig_cm_single(cm, f"{name} — {fs}"), d, f"cm_{tag}_{fslug}_{safe}",
                             "", f"Confusion matrix {name}, {fs}, {tag}")
            print(f"  [{tag}] {fs:<10s} {name:<30s} acc {m[acc_key()]:8.4f}  F1 {m[f1_key()]:8.4f}  "
                  f"AUC {m[auc_key()]:.4f}  train {np.mean(fit_t):8.3f}s  infer {np.mean(inf_t)*1000:9.2f}ms")
        if figures:
            save_fig(fig_cm_grid(cms, f"{DATASET} — {fs} — {tag} (held-out test set)"), d,
                     f"fig_cm_grid_{tag}_{fslug}", f"Confusion-matrix figure ({fs}, {tag})", f"All models, {fs}, {tag}")
            save_fig(fig_roc(curves, f"ROC curves — {DATASET}, {fs} ({tag})" +
                             ("" if N_CLASSES == 2 else "\nmicro-average curve; legend AUC = macro one-vs-rest")),
                     d, f"fig_roc_{tag}_{fslug}", f"ROC figure ({fs}, {tag})", f"All models, {fs}, {tag}")
    return pd.DataFrame(rows), preds, selected


# ---------------- 10-fold CV ----------------
def cross_validate_models(X, y, tr, num_cols, cat_cols, model_dict, feature_sets, folds=10):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    Xtr = X.iloc[tr].reset_index(drop=True); ytr = np.asarray(y)[tr]
    rows = []
    for fs, k in feature_sets.items():
        for name, base in model_dict.items():
            acc, prec, rec, f1w, auc = [], [], [], [], []
            for a, b in skf.split(Xtr, ytr):
                pipe = make_pipeline(num_cols, cat_cols, clone(base), k).fit(Xtr.iloc[a], ytr[a])
                pr = pipe.predict(Xtr.iloc[b]); s = model_scores(pipe.named_steps["clf"], Pipeline(pipe.steps[:-1]).transform(Xtr.iloc[b]))
                yt = ytr[b]
                acc.append(accuracy_score(yt, pr) * 100)
                prec.append(precision_score(yt, pr, average="weighted", zero_division=0) * 100)
                rec.append(recall_score(yt, pr, average="weighted", zero_division=0) * 100)
                f1w.append(f1_score(yt, pr, average="weighted", zero_division=0) * 100)
                auc.append((roc_auc_score(yt, s) if N_CLASSES == 2 else ovr_auc(yt, s)) * 100)
            rows.append({"Feature set": fs, "Model": name, "Folds": folds,
                         "Accuracy mean (%)": np.mean(acc), "Accuracy SD": np.std(acc, ddof=1),
                         "Weighted precision mean (%)": np.mean(prec), "Weighted recall mean (%)": np.mean(rec),
                         "Weighted F1 mean (%)": np.mean(f1w), "Weighted F1 SD": np.std(f1w, ddof=1),
                         "ROC-AUC mean (%)": np.mean(auc), "ROC-AUC SD": np.std(auc, ddof=1)})
            print(f"  CV {fs:<10s} {name:<22s} {np.mean(acc):7.3f} ± {np.std(acc, ddof=1):.3f}")
    return pd.DataFrame(rows)


def cv_paper_table(cv, fs):
    d = cv[cv["Feature set"] == fs]
    return pd.DataFrame({
        "Model": d["Model"],
        "Accuracy (Mean ± SD)": [f"{a:.2f} ± {b:.2f}" for a, b in zip(d["Accuracy mean (%)"], d["Accuracy SD"])],
        "Weighted Precision (%)": d["Weighted precision mean (%)"].round(2).values,
        "Weighted Recall (%)": d["Weighted recall mean (%)"].round(2).values,
        "Weighted F1 (Mean ± SD)": [f"{a:.2f} ± {b:.2f}" for a, b in zip(d["Weighted F1 mean (%)"], d["Weighted F1 SD"])],
        "ROC-AUC (%)": d["ROC-AUC mean (%)"].round(2).values})


# ---------------- McNemar (all pairs, Holm) ----------------
def mcnemar_all(y_te, preds, feature_sets, model_names):
    rows = []
    for fs in feature_sets:
        tmp = []
        for m1, m2 in itertools.combinations(model_names, 2):
            b, c, st, pv, test = mcnemar(y_te, preds[(fs, m1)], preds[(fs, m2)])
            tmp.append({"Feature set": fs, "Model 1": m1, "Model 2": m2,
                        "b (Model 1 correct, Model 2 wrong)": b, "c (Model 1 wrong, Model 2 correct)": c,
                        "Statistic": st, "p-value": pv, "Test": test})
        adj = holm([t["p-value"] for t in tmp])
        for t, a in zip(tmp, adj):
            t["p-value (Holm)"] = a; t["Significant at 0.05 (Holm)"] = "Yes" if a < 0.05 else "No"
        rows += tmp
    return pd.DataFrame(rows)


# ---------------- comparison figures (Raw vs MI) ----------------
def comparison_outputs(res, fs_new, tag_fig_acc, tag_fig_f1, tag_fig_cost, tag_fig_heat, paper_tab):
    a = res[res["Feature set"] == "RAW"].set_index("Model")
    b = res[res["Feature set"] == fs_new].set_index("Model")
    tab = pd.DataFrame({
        "Accuracy Raw (%)": a[acc_key()], f"Accuracy {fs_new} (%)": b[acc_key()],
        "Δ Accuracy (pp)": b[acc_key()] - a[acc_key()],
        "Weighted F1 Raw (%)": a["Weighted F1 (%)"], f"Weighted F1 {fs_new} (%)": b["Weighted F1 (%)"],
        "Δ Weighted F1 (pp)": b["Weighted F1 (%)"] - a["Weighted F1 (%)"],
        "Training Raw (s)": a["Training time mean (s)"], f"Training {fs_new} (s)": b["Training time mean (s)"],
        "Inference Raw (ms)": a["Inference time mean (s)"] * 1000,
        f"Inference {fs_new} (ms)": b["Inference time mean (s)"] * 1000,
        "Preprocessing+MI Raw (s)": a["Preprocessing+MI fit (s)"],
        f"Preprocessing+MI {fs_new} (s)": b["Preprocessing+MI fit (s)"]}).round(4)
    save_table(tab, "07_comparison", f"raw_vs_{fs_new.replace(' ', '').replace('-', '')}", paper_tab,
               f"Raw vs {fs_new}: accuracy, F1, training, inference, preprocessing", index=True)
    models = list(tab.index); x = np.arange(len(models)); w = 0.38
    # accuracy + time
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    for ax, (c1, c2, lab, log) in zip(axes, [("Accuracy Raw (%)", f"Accuracy {fs_new} (%)", "Accuracy (%)", False),
                                             ("Training Raw (s)", f"Training {fs_new} (s)", "Training time (s, log)", True)]):
        r1 = ax.bar(x - w / 2, tab[c1], w, label="Raw", edgecolor="black")
        r2 = ax.bar(x + w / 2, tab[c2], w, label=fs_new, edgecolor="black")
        ax.bar_label(r1, fmt="%.2f" if not log else "%.3g", fontsize=7, rotation=90, padding=2)
        ax.bar_label(r2, fmt="%.2f" if not log else "%.3g", fontsize=7, rotation=90, padding=2)
        if log: ax.set_yscale("log")
        else: ax.set_ylim(max(0, min(tab[c1].min(), tab[c2].min()) - 10), 104)
        ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right"); ax.set_ylabel(lab)
        ax.legend(); ax.grid(axis="y", alpha=0.3)
    axes[0].set_title(f"Accuracy: Raw vs {fs_new}"); axes[1].set_title(f"Training time: Raw vs {fs_new}")
    fig.tight_layout(); save_fig(fig, "07_comparison", f"fig_accuracy_time_{fs_new.replace(' ', '')}", tag_fig_acc,
                                 f"Accuracy and training time, Raw vs {fs_new}")
    # F1 change
    fig, ax = plt.subplots(figsize=(9, 4.4))
    dlt = tab["Δ Weighted F1 (pp)"]
    bars = ax.bar(models, dlt, color=["#2ca02c" if v >= 0 else "#d62728" for v in dlt], edgecolor="black")
    ax.bar_label(bars, fmt="%+.2f", fontsize=8); ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("Δ weighted F1 (percentage points)")
    ax.set_title(f"Change in weighted-average F1-score: {fs_new} − Raw"); ax.grid(axis="y", alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right"); fig.tight_layout()
    save_fig(fig, "07_comparison", f"fig_f1_change_{fs_new.replace(' ', '')}", tag_fig_f1, f"Δ weighted F1, {fs_new}")
    # computational cost: stacked preprocessing / training / inference
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for off, src, lab in [(-w / 2, a, "Raw"), (w / 2, b, fs_new)]:
        p1 = src["Preprocessing+MI fit (s)"].values; t1 = src["Training time mean (s)"].values
        i1 = src["Inference time mean (s)"].values
        ax.bar(x + off, p1, w, label=f"{lab}: preprocessing+MI", edgecolor="black", alpha=0.6)
        ax.bar(x + off, t1, w, bottom=p1, label=f"{lab}: training", edgecolor="black")
        ax.bar(x + off, i1, w, bottom=p1 + t1, label=f"{lab}: inference", edgecolor="black", hatch="//")
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right")
    ax.set_ylabel("Time (s, log scale)"); ax.set_title(f"Computational cost: Raw vs {fs_new}")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left"); ax.grid(axis="y", alpha=0.3); fig.tight_layout()
    save_fig(fig, "07_comparison", f"fig_cost_{fs_new.replace(' ', '')}", tag_fig_cost,
             f"Preprocessing+MI, training and inference time, Raw vs {fs_new}")
    # heatmap
    hm = pd.DataFrame({"Acc Raw": tab["Accuracy Raw (%)"], f"Acc {fs_new}": tab[f"Accuracy {fs_new} (%)"],
                       "Wtd F1 Raw": tab["Weighted F1 Raw (%)"], f"Wtd F1 {fs_new}": tab[f"Weighted F1 {fs_new} (%)"]}).T
    fig, ax = plt.subplots(figsize=(10, 3.6))
    im = ax.imshow(hm.values, cmap="YlGnBu", aspect="auto", vmin=min(60, hm.values.min()), vmax=100)
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            ax.text(j, i, f"{hm.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns, rotation=20, ha="right")
    ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index); fig.colorbar(im, ax=ax, label="%")
    ax.set_title(f"Accuracy and weighted F1 (%): Raw vs {fs_new}"); fig.tight_layout()
    save_fig(fig, "07_comparison", f"fig_heatmap_{fs_new.replace(' ', '')}", tag_fig_heat,
             f"Heatmap of accuracy and weighted F1, Raw vs {fs_new}")
    return tab


def cv_figure(cv, order, paper):
    fig, axes = plt.subplots(1, len(order), figsize=(5.2 * len(order), 4.6), sharey=False)
    for ax, fs in zip(np.atleast_1d(axes), order):
        d = cv[cv["Feature set"] == fs]
        bars = ax.bar(d["Model"], d["Accuracy mean (%)"], yerr=d["Accuracy SD"], capsize=4, edgecolor="black")
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=3)
        ax.set_ylim(max(0, d["Accuracy mean (%)"].min() - 8), 101); ax.set_title(f"10-fold CV accuracy — {fs}")
        ax.set_ylabel("Accuracy (%)"); plt.setp(ax.get_xticklabels(), rotation=25, ha="right"); ax.grid(axis="y", alpha=0.3)
    fig.tight_layout(); save_fig(fig, "08_cross_validation", "fig_cv_accuracy", paper, "10-fold CV accuracy (mean ± SD)")


def timing_outputs(res, order, paper_tab, paper_lat, paper_thr):
    t = res[["Feature set", "Model", "N features", "Preprocessing+MI fit (s)", "Transform train+test (s)",
             "Training time mean (s)", "Training time SD (s)", "Training repeats", "Inference time mean (s)",
             "Inference time SD (s)", "Latency (ms/sample)", "Throughput (samples/s)"]].round(6)
    save_table(t, "10_timing", "timing", paper_tab, "Preprocessing, training and inference time (mean ± SD), latency, throughput")
    save_fig(fig_grouped_bars(res, "Latency (ms/sample)", "Latency (ms/sample, log)", "Inference latency", order, log=True),
             "10_timing", "fig_latency", paper_lat, "Inference latency by feature set")
    save_fig(fig_grouped_bars(res, "Throughput (samples/s)", "Throughput (samples/s, log)", "Inference throughput", order, log=True),
             "10_timing", "fig_throughput", paper_thr, "Inference throughput by feature set")
    save_fig(fig_grouped_bars(res, "Training time mean (s)", "Training time (s, log)", "Training time", order, log=True),
             "10_timing", "fig_training_time", "", "Training time by feature set")


def mi_outputs(X, y, tr, num_cols, cat_cols, paper_prefix):
    prep = make_pipeline(num_cols, cat_cols, None, None)
    Xt = Pipeline(prep.steps[:-1]).fit(X.iloc[tr], np.asarray(y)[tr]).transform(X.iloc[tr])
    mi = mutual_info_classif(np.asarray(Xt, float), np.asarray(y)[tr], random_state=SEED)
    df = pd.DataFrame({"Feature": list(num_cols) + list(cat_cols), "MI score": mi})
    df = df.sort_values("MI score", ascending=False, kind="stable").reset_index(drop=True)
    df["Rank"] = np.arange(1, len(df) + 1)
    df["In Top-5"] = df["Rank"] <= 5; df["In Top-12"] = df["Rank"] <= 12
    save_table(df, "03_mutual_information", "mi_ranking", f"{paper_prefix} MI ranking table",
               "MI score of every input feature (training data)")
    fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(df)), 4.6))
    cols = ["#1f77b4" if r <= 5 else "#6baed6" if r <= 12 else "#c6dbef" for r in df["Rank"]]
    ax.bar(df["Feature"], df["MI score"], color=cols, edgecolor="black")
    ax.set_ylabel("MI score"); ax.set_title("Mutual information of each feature (training data; dark = Top-5, mid = Top-12)")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right"); ax.grid(axis="y", alpha=0.3); fig.tight_layout()
    save_fig(fig, "03_mutual_information", "fig_mi_scores", f"{paper_prefix} Figure 3", "MI scores, all features")
    fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(df)), 1.9))
    im = ax.imshow(df[["MI score"]].T.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(df))); ax.set_xticklabels(df["Feature"], rotation=45, ha="right"); ax.set_yticks([])
    for j, v in enumerate(df["MI score"]):
        ax.text(j, 0, f"{v:.3f}", ha="center", va="center", fontsize=7, color="white" if v < df["MI score"].max() * 0.6 else "black")
    fig.colorbar(im, ax=ax, fraction=0.03); ax.set_title("Feature importance heatmap (MI score)"); fig.tight_layout()
    save_fig(fig, "03_mutual_information", "fig_mi_heatmap", f"{paper_prefix} Figure 4", "MI heatmap")
    for name, part, title, paper in [("fig_mi_top10", df.head(10), "Top-10 features by MI", "Figure 5"),
                                     ("fig_mi_least", df.tail(max(1, len(df) - 12)).iloc[::-1],
                                      "Features outside the Top-12 (least informative)", "Figure 6")]:
        fig, ax = plt.subplots(figsize=(7, 0.35 * len(part) + 1.2))
        ax.barh(part["Feature"][::-1] if name == "fig_mi_top10" else part["Feature"],
                part["MI score"][::-1] if name == "fig_mi_top10" else part["MI score"], edgecolor="black")
        ax.set_xlabel("MI score"); ax.set_title(title); ax.grid(axis="x", alpha=0.3); fig.tight_layout()
        save_fig(fig, "03_mutual_information", name, f"{paper_prefix} {paper}", title)
    return df


def descriptive_outputs(X, num_cols, tr, paper_tab, paper_fig):
    desc = X[num_cols].describe().T[["mean", "std", "25%", "50%", "75%", "max"]].round(2)
    save_table(desc, "02_descriptive", "descriptive_statistics", paper_tab, "Descriptive statistics of numeric inputs", index=True)
    nonconst = [c for c in num_cols if X[c].iloc[tr].nunique() > 1]
    corr = X[nonconst].iloc[tr].corr()
    fig, ax = plt.subplots(figsize=(0.55 * len(nonconst) + 3, 0.5 * len(nonconst) + 2))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    for i in range(len(nonconst)):
        for j in range(len(nonconst)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
    ax.set_xticks(range(len(nonconst))); ax.set_xticklabels(nonconst, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(nonconst))); ax.set_yticklabels(nonconst, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.04); ax.set_title("Pearson correlation of numeric inputs (training data)")
    fig.tight_layout(); save_fig(fig, "02_descriptive", "fig_correlation", paper_fig, "Correlation heatmap")
    save_table(corr.round(4), "02_descriptive", "correlation_matrix", "", "Correlation values", index=True)


def vector_audit(X, y, tr, te, label=""):
    v = X.astype(str).agg("|".join, axis=1)
    g = pd.DataFrame({"v": v.values, "y": np.asarray(y)})
    nlab = g.groupby("v")["y"].nunique()
    major = g.groupby("v")["y"].agg(lambda s: s.value_counts().iloc[0]).sum()
    trv = set(v.iloc[tr])
    return {"Feature set": label, "Features": X.shape[1], "Records": len(X),
            "Duplicate records (features + label)": int(pd.concat([X.reset_index(drop=True), pd.Series(np.asarray(y), name='__y')], axis=1).duplicated().sum()),
            "Duplicate records (features only)": int(X.duplicated().sum()),
            "Distinct feature vectors": int(v.nunique()),
            "Vectors with more than one label": int((nlab > 1).sum()),
            "Accuracy ceiling (%)": round(major / len(g) * 100, 4),
            "Test records whose vector occurs in training (%)": round(v.iloc[te].isin(trv).mean() * 100, 4)}


def export_all():
    pm = pd.DataFrame(PAPER_MAP, columns=["Output file", "Paper item", "Description"])
    pm.to_csv(p("13_export", "paper_map.csv"), index=False)
    with pd.ExcelWriter(p("13_export", f"{DATASET.replace(' ', '_')}_all_tables.xlsx")) as xw:
        for name, df in TABLES.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
        pm.to_excel(xw, sheet_name="paper_map", index=False)
    zpath = os.path.join(os.path.dirname(os.path.abspath(OUT)) or ".", f"{os.path.basename(OUT)}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUT):
            for f in files:
                full = os.path.join(root, f); z.write(full, os.path.relpath(full, os.path.dirname(os.path.abspath(OUT))))
    print(f"\nExported {len(TABLES)} tables and {sum(1 for r in PAPER_MAP if r[0].endswith('.png'))} figures.")
    print(f"Paper map: {p('13_export', 'paper_map.csv')}\nZip for download: {zpath}")

## 0. Environment

In [3]:
setup_dirs()
ENV = environment()

                      Item                                   Value
                       CPU          Intel(R) Xeon(R) CPU @ 2.20GHz
              Logical CPUs                                       4
                       RAM                                 31.3 GB
                        OS    Linux-6.18.48+-x86_64-with-glibc2.39
                    Python                                 3.13.15
              scikit-learn                                   1.6.1
                   XGBoost                                   3.4.1
                     NumPy                                   2.1.3
                    pandas                                   2.3.3
Threads per model (n_jobs)                                       4
                    Device CPU only; no GPU used by any classifier
               Random seed                                      42
     Training-time repeats                                       3
    Inference-time repeats                                    

## 1. Dataset audit and input features

In [4]:
raw = pd.read_csv(DATA_PATH)
print("File:", raw.shape, "\nColumns:", list(raw.columns))
n_missing = int(raw.isna().any(axis=1).sum())
df = raw.dropna().reset_index(drop=True)
y = df[TARGET].astype(int).values
X = df.drop(columns=[TARGET] + [c for c in ID_COLS if c in df.columns])
cat_cols = list(X.select_dtypes(include=["object", "category", "bool"]).columns)
num_cols = [c for c in X.columns if c not in cat_cols]
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
y_te = y[te]
print(f"Records used {len(df):,} | train {len(tr):,} | test {len(te):,} | inputs {X.shape[1]}")

audit = pd.DataFrame([
    ("Records in file", len(raw)), ("Records with missing values (removed)", n_missing),
    ("Records used", len(df)),
    ("Benign / attack (all)", f"{(y == 0).sum():,} / {(y == 1).sum():,}"),
    ("Benign / attack (train)", f"{(y[tr] == 0).sum():,} / {(y[tr] == 1).sum():,}"),
    ("Benign / attack (test)", f"{(y_te == 0).sum():,} / {(y_te == 1).sum():,}"),
    ("Exact duplicate rows (all 23 columns)", int(raw.duplicated().sum())),
], columns=["Quantity", "Value"])
va = vector_audit(X, y, tr, te, "19 input features")
audit = pd.concat([audit, pd.DataFrame([(k, v) for k, v in va.items() if k not in ("Feature set",)],
                                       columns=["Quantity", "Value"])], ignore_index=True)
save_table(audit, "01_dataset_audit", "dataset_audit", "New Table (dataset audit)",
           "Missing values, class balance, duplicates, distinct vectors, train/test overlap")
print(audit.to_string(index=False))

feat_tab = pd.DataFrame({"Attribute": list(raw.columns)})
feat_tab["Type"] = [str(raw[c].dtype) for c in raw.columns]
feat_tab["Role"] = ["Label" if c == TARGET else "Excluded (identifier: time / host)" if c in ID_COLS
                    else "Input (topology)" if c in TOPOLOGY_COLS else "Input" for c in raw.columns]
save_table(feat_tab, "01_dataset_audit", "input_features", "Section 3.1 feature list", "All attributes and their role")
print(feat_tab.to_string(index=False))

File: (104345, 23) 
Columns: ['dt', 'switch', 'src', 'dst', 'pktcount', 'bytecount', 'dur', 'dur_nsec', 'tot_dur', 'flows', 'packetins', 'pktperflow', 'byteperflow', 'pktrate', 'Pairflow', 'Protocol', 'port_no', 'tx_bytes', 'rx_bytes', 'tx_kbps', 'rx_kbps', 'tot_kbps', 'label']
Records used 103,839 | train 72,687 | test 31,152 | inputs 19
                                        Quantity           Value
                                 Records in file          104345
           Records with missing values (removed)             506
                                    Records used          103839
                           Benign / attack (all) 63,335 / 40,504
                         Benign / attack (train) 44,334 / 28,353
                          Benign / attack (test) 19,001 / 12,151
           Exact duplicate rows (all 23 columns)            5091
                                        Features            19.0
                                         Records        103839.0
         

## 2. Descriptive statistics and correlation

In [5]:
descriptive_outputs(X, num_cols, tr, "Table 6", "Figure 7")

## 3. Mutual-information ranking (training data only)

In [6]:
MI = mi_outputs(X, y, tr, num_cols, cat_cols, "")
print(MI.to_string(index=False))

    Feature  MI score  Rank  In Top-5  In Top-12
  bytecount  0.602605     1      True       True
   pktcount  0.582220     2      True       True
byteperflow  0.556218     3      True       True
 pktperflow  0.541048     4      True       True
    pktrate  0.385283     5      True       True
    tot_dur  0.252188     6     False       True
        dur  0.196778     7     False       True
  packetins  0.193590     8     False       True
   tx_bytes  0.190589     9     False       True
   rx_bytes  0.161906    10     False       True
   tot_kbps  0.102908    11     False       True
    rx_kbps  0.068730    12     False       True
    tx_kbps  0.068526    13     False      False
   dur_nsec  0.067481    14     False      False
   Protocol  0.041415    15     False      False
      flows  0.028871    16     False      False
   Pairflow  0.010384    17     False      False
     switch  0.004525    18     False      False
    port_no  0.002706    19     False      False


## 4–6. Six classifiers on Raw, MI Top-5 and MI Top-12 (held-out test set)

In [7]:
RES, PREDS, SELECTED = evaluate(X, y, tr, te, num_cols, cat_cols, models_main(), FEATURE_SETS, "main", FS_DIR)
json.dump(SELECTED, open(p("03_mutual_information", "selected_features.json"), "w"), indent=2)
print("\nSelected features:", json.dumps(SELECTED, indent=1))
save_table(RES.round(6), "04_raw", "test_results_all_feature_sets", "Tables 7-9 source", "Every metric, every model, every feature set")
cols7 = ["Model", "Accuracy (%)", "Precision (%)", "Recall (%)", "Specificity (%)", "F1 (%)",
         "Weighted F1 (%)", "ROC-AUC", "PR-AUC", "TN", "FP", "FN", "TP"]
for fs, d in FS_DIR.items():
    t = RES[RES["Feature set"] == fs][cols7].round(4)
    save_table(t, d, f"test_metrics_{fs.replace(' ', '').replace('-', '')}",
               "Table 7" if fs == "RAW" else f"Test results {fs}", f"Held-out test metrics, {fs}")
    print(f"\n{fs}\n", t.to_string(index=False))

  [main] RAW        Logistic Regression            acc  76.6403  F1  68.8791  AUC 0.8375  train    0.469s  infer      1.08ms
  [main] RAW        SVM                            acc  96.8573  F1  96.0033  AUC 0.9947  train   44.236s  infer  15158.36ms
  [main] RAW        KNN                            acc  98.4656  F1  98.0307  AUC 0.9974  train    0.006s  infer  23663.18ms
  [main] RAW        Decision Tree                  acc  98.2024  F1  97.7199  AUC 0.9924  train    0.296s  infer      2.35ms
  [main] RAW        Random Forest                  acc  99.4703  F1  99.3232  AUC 0.9999  train    0.608s  infer     25.49ms
  [main] RAW        XGBoost                        acc 100.0000  F1 100.0000  AUC 1.0000  train    0.539s  infer     29.86ms
  [main] MI Top-5   Logistic Regression            acc  70.8398  F1  58.2767  AUC 0.7655  train    0.069s  infer      0.45ms
  [main] MI Top-5   SVM                            acc  92.5398  F1  90.4582  AUC 0.9739  train   46.330s  infer  20662.60ms


## 7. Raw vs MI Top-5 vs MI Top-12

In [8]:
T8 = comparison_outputs(RES, "MI Top-5", "Figure 10", "Figure 11", "Figure 12", "Figure 13", "Table 8")
T9 = comparison_outputs(RES, "MI Top-12", "Figure 14", "Figure 16", "Figure 15", "Figure 17 (Top-12 heatmap)", "Table 9")
acc_all = RES.pivot_table(index="Model", columns="Feature set", values="Accuracy (%)", sort=False)[list(FEATURE_SETS)].round(2)
save_table(acc_all, "07_comparison", "accuracy_all_feature_sets", "Table 19 (DDoS-SDN only)", "Accuracy by feature set", index=True)
red = pd.DataFrame([{"Feature set": fs, "Input features": X.shape[1], "Selected": len(SELECTED[fs]),
                     "Reduction (%)": round((1 - len(SELECTED[fs]) / X.shape[1]) * 100, 1),
                     "Features": ", ".join(SELECTED[fs])} for fs in FEATURE_SETS])
save_table(red, "07_comparison", "feature_reduction", "Section 4.4/4.5 text", "Dimensionality reduction")
print(T8, "\n", T9, "\n", acc_all, "\n", red)

                     Accuracy Raw (%)  Accuracy MI Top-5 (%)  Δ Accuracy (pp)  Weighted F1 Raw (%)  Weighted F1 MI Top-5 (%)  Δ Weighted F1 (pp)  Training Raw (s)  Training MI Top-5 (s)  Inference Raw (ms)  \
Model                                                                                                                                                                                                           
Logistic Regression           76.6403                70.8398          -5.8006              76.4571                   70.0556             -6.4015            0.4693                 0.0695              1.0784   
SVM                           96.8573                92.5398          -4.3175              96.8617                   92.5428             -4.3189           44.2362                46.3302          15158.3609   
KNN                           98.4656                99.9711           1.5055              98.4652                   99.9711              1.5059            0.0061  

## 8. Stratified 10-fold cross-validation on the training portion
MI selection, imputation, encoding and scaling are refitted inside every training fold.
SVM is excluded from CV only (runtime); it is evaluated on the held-out test set in Sections 4–6.

In [9]:
if RUN_CV:
    cv_models = {k: v for k, v in models_main().items() if k != "SVM"}
    CV = cross_validate_models(X, y, tr, num_cols, cat_cols, cv_models, FEATURE_SETS, CV_FOLDS)
    save_table(CV.round(4), "08_cross_validation", "cv_all", "Tables 10-12 source", "10-fold CV, all metrics")
    for fs, tab in zip(FEATURE_SETS, ["Table 10", "Table 11", "Table 12"]):
        save_table(cv_paper_table(CV, fs), "08_cross_validation", f"cv_{fs.replace(' ', '').replace('-', '')}",
                   tab, f"10-fold CV, {fs} (paper format)")
    cv_figure(CV, list(FEATURE_SETS), "Figure 25")
    print(CV.round(3).to_string(index=False))

  CV RAW        Logistic Regression     76.931 ± 0.543
  CV RAW        KNN                     98.256 ± 0.216
  CV RAW        Decision Tree           98.140 ± 0.186
  CV RAW        Random Forest           99.345 ± 0.111
  CV RAW        XGBoost                100.000 ± 0.000
  CV MI Top-5   Logistic Regression     71.249 ± 0.543
  CV MI Top-5   KNN                     99.927 ± 0.044
  CV MI Top-5   Decision Tree           94.190 ± 0.276
  CV MI Top-5   Random Forest           96.664 ± 0.231
  CV MI Top-5   XGBoost                 99.088 ± 0.101
  CV MI Top-12  Logistic Regression     74.680 ± 0.608
  CV MI Top-12  KNN                     99.227 ± 0.133
  CV MI Top-12  Decision Tree           98.205 ± 0.104
  CV MI Top-12  Random Forest           99.415 ± 0.100
  CV MI Top-12  XGBoost                 99.999 ± 0.004
Feature set               Model  Folds  Accuracy mean (%)  Accuracy SD  Weighted precision mean (%)  Weighted recall mean (%)  Weighted F1 mean (%)  Weighted F1 SD  ROC-AUC me

## 9. McNemar's test on the held-out predictions (all pairs, Holm correction)

In [10]:
MCN = mcnemar_all(y_te, PREDS, FEATURE_SETS, list(models_main()))
save_table(MCN, "09_mcnemar", "mcnemar_all_pairs_holm", "Table 15", "All 15 pairs per feature set, Holm-adjusted")
print(MCN.to_string(index=False))

Feature set             Model 1       Model 2  b (Model 1 correct, Model 2 wrong)  c (Model 1 wrong, Model 2 correct)   Statistic       p-value                                  Test  p-value (Holm) Significant at 0.05 (Holm)
        RAW Logistic Regression           SVM                                 363                                6661 5645.246156  0.000000e+00 chi-square with continuity correction    0.000000e+00                        Yes
        RAW Logistic Regression           KNN                                 216                                7015 6390.928502  0.000000e+00 chi-square with continuity correction    0.000000e+00                        Yes
        RAW Logistic Regression Decision Tree                                 250                                6967 6249.779133  0.000000e+00 chi-square with continuity correction    0.000000e+00                        Yes
        RAW Logistic Regression Random Forest                                  73                   

## 10. Timing: preprocessing + MI, training, inference (CPU only)

In [11]:
timing_outputs(RES, list(FEATURE_SETS), "Tables 16/18 (merged)", "Figure 23", "Figure 24")
print(RES[["Feature set", "Model", "Preprocessing+MI fit (s)", "Training time mean (s)", "Training time SD (s)",
           "Inference time mean (s)", "Inference time SD (s)", "Latency (ms/sample)", "Throughput (samples/s)"]].round(5).to_string(index=False))

Feature set               Model  Preprocessing+MI fit (s)  Training time mean (s)  Training time SD (s)  Inference time mean (s)  Inference time SD (s)  Latency (ms/sample)  Throughput (samples/s)
        RAW Logistic Regression                   0.20015                 0.46929               0.02858                  0.00108                0.00065              0.00003            2.888773e+07
        RAW                 SVM                   0.20015                44.23621                   NaN                 15.15836                0.02106              0.48659            2.055103e+03
        RAW                 KNN                   0.20015                 0.00609               0.00034                 23.66318                0.08227              0.75960            1.316476e+03
        RAW       Decision Tree                   0.20015                 0.29628               0.00200                  0.00235                0.00066              0.00008            1.325020e+07
        RAW    

## 11. Capacity check: unconstrained Decision Tree, Random Forest and XGBoost

In [12]:
UNC, _, _ = evaluate(X, y, tr, te, num_cols, cat_cols, models_unconstrained(), FEATURE_SETS, "unconstrained",
                     {fs: "11_capacity_check" for fs in FEATURE_SETS}, timing=False, figures=False)
cmp_rows = []
for fs in FEATURE_SETS:
    for base, unc in [("Decision Tree", "Decision Tree (unconstrained)"), ("Random Forest", "Random Forest (unconstrained)"),
                      ("XGBoost", "XGBoost (500 trees, depth 10)")]:
        a = RES[(RES["Feature set"] == fs) & (RES["Model"] == base)].iloc[0]
        b = UNC[(UNC["Feature set"] == fs) & (UNC["Model"] == unc)].iloc[0]
        cmp_rows.append({"Feature set": fs, "Model": base, "Accuracy constrained (%)": a["Accuracy (%)"],
                         "Accuracy unconstrained (%)": b["Accuracy (%)"],
                         "Δ (pp)": b["Accuracy (%)"] - a["Accuracy (%)"], "F1 unconstrained (%)": b["F1 (%)"],
                         "ROC-AUC unconstrained": b["ROC-AUC"]})
T20 = pd.DataFrame(cmp_rows).round(4)
save_table(UNC.round(6), "11_capacity_check", "unconstrained_all_metrics", "Table 20 source", "Unconstrained models, all metrics")
save_table(T20, "11_capacity_check", "capacity_check", "Table 20", "Constrained vs unconstrained accuracy")
print(T20.to_string(index=False))

  [unconstrained] RAW        Decision Tree (unconstrained)  acc  99.9647  F1  99.9547  AUC 0.9996  train    0.465s  infer      2.87ms
  [unconstrained] RAW        Random Forest (unconstrained)  acc 100.0000  F1 100.0000  AUC 1.0000  train    6.213s  infer    142.88ms
  [unconstrained] RAW        XGBoost (500 trees, depth 10)  acc 100.0000  F1 100.0000  AUC 1.0000  train    1.598s  infer     95.11ms
  [unconstrained] MI Top-5   Decision Tree (unconstrained)  acc  99.9647  F1  99.9547  AUC 0.9998  train    0.131s  infer      2.92ms
  [unconstrained] MI Top-5   Random Forest (unconstrained)  acc  99.9807  F1  99.9753  AUC 1.0000  train    4.001s  infer    143.82ms
  [unconstrained] MI Top-5   XGBoost (500 trees, depth 10)  acc  99.3901  F1  99.2186  AUC 0.9999  train    1.740s  infer    197.88ms
  [unconstrained] MI Top-12  Decision Tree (unconstrained)  acc  99.9936  F1  99.9918  AUC 0.9999  train    0.411s  infer      2.58ms
  [unconstrained] MI Top-12  Random Forest (unconstrained)  ac

## 12. Robustness: time-ordered split, unseen source hosts, no topology features
dt and src define the splits only; they are never model inputs.

In [13]:
ROB_SETS = {"RAW": None, "MI Top-12": 12}
rob_dirs = {fs: "12_dataset_specific" for fs in ROB_SETS}
ROB = []
# (a) time-ordered: within each class, earliest 70% (by dt) -> train, latest 30% -> test
if "dt" in df.columns:
    tr_t, te_t = [], []
    for cl in (0, 1):
        ii = np.where(y == cl)[0]; ii = ii[np.argsort(df["dt"].values[ii], kind="stable")]
        cut = int(len(ii) * 0.7); tr_t += list(ii[:cut]); te_t += list(ii[cut:])
    r, _, _ = evaluate(X, y, np.array(tr_t), np.array(te_t), num_cols, cat_cols, models_main(), ROB_SETS,
                       "time_ordered", rob_dirs, timing=False)
    ROB.append(r)
# (b) unseen source hosts: source IPs in the test set never appear in training
host_note = "not run"
if "src" in df.columns:
    hosts = df["src"].astype(str).values; uniq = np.unique(hosts); rng = np.random.RandomState(SEED); best = None
    for _ in range(1000):
        perm = rng.permutation(uniq); m = np.zeros(len(hosts), bool)
        for h in perm:
            m |= hosts == h
            if m.mean() >= 0.25:
                break
        ok = all(len(np.unique(y[s])) == 2 for s in (m, ~m))
        if ok and (best is None or abs(m.mean() - 0.30) < abs(best.mean() - 0.30)):
            best = m.copy()
    if best is not None:
        held = sorted(set(hosts[best]))
        host_note = f"{len(held)} of {len(uniq)} source IPs held out ({best.mean()*100:.1f}% of records): {', '.join(held)}"
        r, _, _ = evaluate(X, y, np.where(~best)[0], np.where(best)[0], num_cols, cat_cols, models_main(), ROB_SETS,
                           "unseen_hosts", rob_dirs, timing=False)
        ROB.append(r)
    else:
        host_note = "not feasible: a class would be missing from one side"
print("Unseen-host split:", host_note)
# (c) without topology features, same random split as the main experiment
Xn = X.drop(columns=[c for c in TOPOLOGY_COLS if c in X.columns])
r, _, _ = evaluate(Xn, y, tr, te, [c for c in num_cols if c in Xn.columns], [c for c in cat_cols if c in Xn.columns],
                   models_main(), ROB_SETS, "no_topology", rob_dirs, timing=False)
ROB.append(r)
ROB = pd.concat(ROB, ignore_index=True)
ref = RES[RES["Feature set"].isin(ROB_SETS)].copy(); ref["Split"] = "main"
both = pd.concat([ref, ROB], ignore_index=True)
T_ROB = both.pivot_table(index=["Feature set", "Model"], columns="Split", values="Accuracy (%)", sort=False).round(2)
T_ROB_F1 = both.pivot_table(index=["Feature set", "Model"], columns="Split", values="F1 (%)", sort=False).round(2)
save_table(ROB.round(6), "12_dataset_specific", "robustness_all_metrics", "Robustness table source", "All metrics")
save_table(T_ROB, "12_dataset_specific", "robustness_accuracy", "New robustness table", "Accuracy by split", index=True)
save_table(T_ROB_F1, "12_dataset_specific", "robustness_f1", "New robustness table", "Attack-class F1 by split", index=True)
save_table(pd.DataFrame([("Unseen-host split", host_note)], columns=["Item", "Value"]), "12_dataset_specific",
           "robustness_split_notes", "Robustness table note", "How the splits were formed")
for fs in ROB_SETS:
    d = both[both["Feature set"] == fs]
    fig = fig_grouped_bars(d.rename(columns={"Split": "Feature set_", "Feature set": "_fs"}).rename(columns={"Feature set_": "Feature set"}),
                           "Accuracy (%)", "Accuracy (%)", f"Robustness of test accuracy — {fs}",
                           ["main", "time_ordered", "unseen_hosts", "no_topology"], annotate=False, legend_title="Split")
    save_fig(fig, "12_dataset_specific", f"fig_robustness_{fs.replace(' ', '').replace('-', '')}", "New robustness figure",
             f"Accuracy under the robustness splits, {fs}")
print(T_ROB.to_string())

  [time_ordered] RAW        Logistic Regression            acc  70.3464  F1  50.1565  AUC 0.6635  train    0.354s  infer      1.26ms
  [time_ordered] RAW        SVM                            acc  88.6945  F1  85.3372  AUC 0.9618  train   32.446s  infer  13349.34ms
  [time_ordered] RAW        KNN                            acc  85.8344  F1  80.5552  AUC 0.8711  train    0.005s  infer  23759.97ms
  [time_ordered] RAW        Decision Tree                  acc  80.4385  F1  78.2155  AUC 0.9006  train    0.255s  infer      2.04ms
  [time_ordered] RAW        Random Forest                  acc  91.1148  F1  87.7511  AUC 0.9861  train    0.482s  infer     14.84ms
  [time_ordered] RAW        XGBoost                        acc  93.3297  F1  90.7406  AUC 0.9484  train    0.400s  infer     29.40ms
  [time_ordered] MI Top-12  Logistic Regression            acc  72.7474  F1  58.0658  AUC 0.7914  train    0.153s  infer      1.07ms
  [time_ordered] MI Top-12  SVM                            acc  86.38

## 13. Pipeline figure (Figure 2) and export

In [14]:
def pipeline_figure():
    steps = ["DDoS-SDN records\n(dt, src, dst\nremoved)", "Stratified\n70/30 split\n(seed 42)",
             "Fit on training\ndata only: impute,\nencode, MI top-k,\nscale", "Train classifier\n(Section 3.2\nsettings)",
             "Evaluate once\non held-out\ntest set", "Metrics,\nMcNemar,\ntiming"]
    fig, ax = plt.subplots(figsize=(14, 2.6)); ax.axis("off")
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(plt.Rectangle((x, 0), 2.0, 1.2, fc="#deebf7", ec="#08519c", lw=1.5))
        ax.text(x + 1.0, 0.6, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.3, 0.6), xytext=(x + 2.0, 0.6), arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text(4.6 + 1.0, -0.35, "10-fold CV on the training portion repeats these steps inside every fold",
            ha="center", fontsize=8, style="italic")
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.6, 1.4)
    return fig

save_fig(pipeline_figure(), "13_export", "fig_pipeline", "Figure 2", "Experimental pipeline")
export_all()


Exported 26 tables and 92 figures.
Paper map: /kaggle/working/DDoS_SDN_results/13_export/paper_map.csv
Zip for download: /kaggle/working/DDoS_SDN_results.zip
